# gesn-search: генерация данных, дообучение и оценка

Весь пайплайн на одной GPU-сессии Kaggle:
1. открытая LLM (Qwen2.5-7B-Instruct) пишет по 5 запросов к каждой норме ГЭСН;
2. запросы чистятся и превращаются в тройки «запрос — норма — трудный негатив»;
3. `multilingual-e5-base` дообучается на тройках;
4. все подходы считаются на ручном тесте (150 запросов) и синтетическом dev.

**Перед запуском** (справа, Session options): Accelerator — **GPU T4 x2**, Internet — **On**.
Затем **Run All**. Полный прогон — примерно 1,5–2 часа; вкладку можно закрыть, если запустить через **Save Version → Save & Run All**.

Результаты в Output: `results_test.md`, `results_dev.md`, `train.jsonl`, `dev.jsonl`, `synth_raw.jsonl`, `e5-gesn.zip`.

In [1]:
# True — быстрый прогон на 30 нормах (~15 мин), чтобы проверить, что всё работает
SMOKE = False
LIMIT = "--limit 30" if SMOKE else ""

In [2]:
!rm -rf /tmp/gesn-search && git clone -q https://github.com/malma0/gesn-search /tmp/gesn-search
%cd /tmp/gesn-search
!pip install -q -e . --no-deps
!pip install -q "rank-bm25>=0.2.2" "pymorphy3>=2.0" "sentence-transformers>=5.0" "accelerate>=1.1.0" datasets
!nvidia-smi --query-gpu=name,memory.total --format=csv

/tmp/gesn-search
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gesn-search (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 61.3 MB/s eta 0:00:00
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 1. Генерация запросов

In [3]:
!python -m gesn.generate {LIMIT}

Норм: 2795, уже готово: 0, осталось: 2795
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 2.83MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 18.8MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 48.0MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 130MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 165MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100%|████████| 27.8k/27.8k [00:00<00:00, 62.1MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 4 files:   0%|                                   | 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|        |  0.00B / 7.73GB            
Reconstructing (incomplete total...):   0%|        |  0.00B / 7.73GB            
Reconstructing (incomplete total...):   0%|        |  0.00B / 15.2GB            
R

## 2. Очистка и тройки

In [4]:
!python -m gesn.synth
!head -c 1500 data/train/train.jsonl

Норм: 2795, сырых запросов: 13910
Отброшено: {'длина': 384, 'копия названия': 20, 'дубль внутри нормы': 48, 'запрещённые слова': 1, 'неоднозначные': 2791}
train: 10134 троек; dev: 532 запросов
{"query": "обои на потолок", "code": "15-06-001-07", "positive": "Отделочные работы. Оклейка обоями потолков. Ед. изм.: 100 м2", "negative": "Отделочные работы. Обивка дверей облицовочными материалами по войлоку. Ед. изм.: 100 м2"}
{"query": "поменять обои в квартире", "code": "15-06-001-07", "positive": "Отделочные работы. Оклейка обоями потолков. Ед. изм.: 100 м2", "negative": "Отделочные работы. Обивка дверей облицовочными материалами по войлоку. Ед. изм.: 100 м2"}
{"query": "прораб обои потолки", "code": "15-06-001-07", "positive": "Отделочные работы. Оклейка обоями потолков. Ед. изм.: 100 м2", "negative": "Отделочные работы. Оклейка обоями стен по монолитной штукатурке и бетону: простыми и средней плотности. Ед. изм.: 100 м2"}
{"query": "клеим обойную пленку на потолок", "code": "15-06-001-0

## 3. Дообучение

In [5]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.train

/tmp/gesn-search/src/gesn/train.py:26: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/gesn-search/src/gesn/train.py:32: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers
Троек: 10134
modules.json: 100%|████████████████████████████| 387/387 [00:00<00:00, 1.45MB/s]
README.md: 100%|█████████████████████████████| 179k/179k [00:00<00:00, 66.3MB/s]
sentence_bert_config.json: 100%|██████████████| 57.0/57.0 [00:00<00:00, 321kB/s]
config.json: 100%|█████████████████████████████| 694/694 [00:00<00:00, 3.80MB/s]

model.safetensors: downloading bytes: ███                   |  153MB, 9.88MB/s  
m

## 4. Оценка

In [6]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.evaluate --methods bm25 e5 hybrid ft=models/e5-gesn bm25+ft --out results_test.md

Batches: 100%|██████████████████████████████████| 92/92 [00:26<00:00,  3.41it/s]
Набор: `data/test/qrels.jsonl`, запросов: 138

### Точный код нормы

| Подход | R@1 | R@5 | R@10 | MRR@10 |
|---|---|---|---|---|
| bm25 | 0.225 [0.15–0.30] | 0.413 [0.33–0.49] | 0.442 [0.36–0.53] | 0.294 [0.23–0.36] |
| e5 | 0.370 [0.29–0.46] | 0.565 [0.49–0.65] | 0.645 [0.57–0.72] | 0.456 [0.38–0.53] |
| bm25+e5 | 0.304 [0.23–0.38] | 0.522 [0.43–0.60] | 0.623 [0.54–0.70] | 0.402 [0.33–0.47] |
| ft | 0.514 [0.43–0.60] | 0.674 [0.60–0.75] | 0.768 [0.70–0.84] | 0.589 [0.52–0.67] |
| bm25+ft | 0.384 [0.30–0.46] | 0.616 [0.54–0.70] | 0.688 [0.61–0.76] | 0.482 [0.41–0.55] |

### Таблица

| Подход | R@1 | R@5 | R@10 | MRR@10 |
|---|---|---|---|---|
| bm25 | 0.246 [0.17–0.32] | 0.507 [0.42–0.59] | 0.609 [0.52–0.69] | 0.366 [0.30–0.44] |
| e5 | 0.406 [0.33–0.49] | 0.681 [0.60–0.76] | 0.746 [0.67–0.82] | 0.522 [0.45–0.59] |
| bm25+e5 | 0.355 [0.28–0.43] | 0.674 [0.59–0.75] | 0.783 [0.71–0.85] | 0.493 [0.42–0.56] |

In [7]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.evaluate --qrels data/train/dev.jsonl --methods bm25 e5 ft=models/e5-gesn --out results_dev.md

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2570.09it/s]
Набор: `data/train/dev.jsonl`, запросов: 532

### Точный код нормы

| Подход | R@1 | R@5 | R@10 | MRR@10 |
|---|---|---|---|---|
| bm25 | 0.141 [0.11–0.17] | 0.387 [0.35–0.43] | 0.483 [0.44–0.52] | 0.246 [0.22–0.28] |
| e5 | 0.203 [0.17–0.24] | 0.457 [0.42–0.50] | 0.566 [0.52–0.61] | 0.312 [0.28–0.34] |
| ft | 0.263 [0.23–0.30] | 0.598 [0.56–0.64] | 0.694 [0.65–0.73] | 0.403 [0.37–0.44] |

### Таблица

| Подход | R@1 | R@5 | R@10 | MRR@10 |
|---|---|---|---|---|
| bm25 | 0.335 [0.29–0.37] | 0.575 [0.53–0.61] | 0.654 [0.61–0.69] | 0.435 [0.40–0.47] |
| e5 | 0.415 [0.37–0.46] | 0.656 [0.62–0.70] | 0.744 [0.71–0.78] | 0.517 [0.48–0.55] |
| ft | 0.541 [0.50–0.58] | 0.795 [0.76–0.83] | 0.859 [0.83–0.89] | 0.652 [0.62–0.69] |



## 5. Сохранение результатов

In [8]:
import shutil
from pathlib import Path

out = Path("/kaggle/working")
for f in ["results_test.md", "results_dev.md", "data/train/train.jsonl", "data/train/dev.jsonl", "data/train/synth_raw.jsonl"]:
    shutil.copy(f, out / Path(f).name)
shutil.make_archive(str(out / "e5-gesn"), "zip", "models/e5-gesn")
print(sorted(p.name for p in out.iterdir()))

['__notebook__.ipynb', 'dev.jsonl', 'e5-gesn.zip', 'results_dev.md', 'results_test.md', 'synth_raw.jsonl', 'train.jsonl']
